# API y dashboard del predictor de flujo de efectivo

Este cuaderno levanta el backend (FastAPI) y el dashboard del repositorio dentro de Colab, sin instalar nada en la computadora. El backend lee los movimientos de un archivo JSON, revisa que tengan todo lo necesario y entrega, para cada serie y modelo, los cuantiles P10, P50 y P90 de los siguientes días hábiles, la alerta de déficit y las métricas de confianza del backtest. La salida llega de dos formas: el dashboard en el navegador y un archivo JSON con el pronóstico completo.

Pasos:

1. Trae el repositorio: lo clona de GitHub o, si todavía no está publicado, pide subir `predictor-flujo-efectivo.zip`.
2. Instala las dependencias del backend (no hace falta GPU ni Chronos).
3. Valida el JSON de movimientos y arranca la API en segundo plano.
4. Consulta la API desde Python, igual que lo haría otro sistema.
5. Revisa el archivo JSON de salida y descárgalo.
6. Abre el dashboard dentro de Colab y en otra pestaña.
7. Corre las pruebas de validación y de contrato.
8. Si cambiaron los datos, vuelve a generar la página publicada en GitHub Pages.

In [ ]:
import os, pathlib, subprocess, zipfile

REPO_URL = "https://github.com/<usuario>/predictor-flujo-efectivo.git"   # la cuenta donde está el repositorio
RAIZ = pathlib.Path("/content/predictor-flujo-efectivo")

if not RAIZ.exists():
    if "<" not in REPO_URL:                      # ya tiene un usuario real: clonar de GitHub
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(RAIZ)], check=True)
    else:
        from google.colab import files
        print("Sube predictor-flujo-efectivo.zip")
        subido = next(iter(files.upload()))
        with zipfile.ZipFile(subido) as z:
            z.extractall("/content")
os.chdir(RAIZ)
print("Repositorio en", RAIZ)
print(sorted(p.name for p in RAIZ.iterdir()))

In [ ]:
%pip install -q -r requirements-api.txt

## Validar la entrada y arrancar la API

La API lee `data/movimientos_feban_sep2025_sep2026_anon.json`. Antes de arrancarla conviene revisar el archivo: la validación dice en qué movimiento y en qué campo falta algo. Para usar otro archivo, súbelo y cambia `ARCHIVO_DATOS`; si viene de un Excel FEBAN, conviértelo antes con `!python src/excel_a_json.py export.xlsx data/movimientos.json`.

In [ ]:
ARCHIVO_DATOS = "data/movimientos_feban_sep2025_sep2026_anon.json"
!python -m api.validar {ARCHIVO_DATOS}

`uvicorn` corre en un proceso aparte, así que las celdas siguientes siguen disponibles. El registro del servidor queda en `api.log`. Con los datos y el backtest del repositorio, el arranque tarda unos segundos; con otro archivo (variable `FLUJO_DATOS`), el backtest de cada modelo se calcula en segundo plano y las métricas aparecen cuando terminan. Si el JSON no pasa la validación, la API no arranca y `api.log` explica por qué.

In [ ]:
import sys, time, requests

PUERTO = 8000
BASE = f"http://127.0.0.1:{PUERTO}/api/v1"
entorno = dict(os.environ, FLUJO_MODELO="ETS", FLUJO_DATOS=ARCHIVO_DATOS)
servidor = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", str(PUERTO)],
    stdout=open("api.log", "w"), stderr=subprocess.STDOUT, env=entorno,
)
for _ in range(120):
    try:
        salud = requests.get(f"{BASE}/salud", timeout=2).json()
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("La API no respondió:\n" + open("api.log").read()[-3000:])
print(f"API {salud['version']} lista con {salud['datos']['archivo']} "
      f"({salud['datos']['movimientos']:,} movimientos, del {salud['datos']['desde']} al {salud['datos']['hasta']})")
print("Modelos:", ", ".join(f"{m['nombre']} (backtest {m['backtest']})" for m in salud["modelos"]))

## Consultar la API como otro sistema

Estas son las mismas peticiones que hace el dashboard. Los montos llegan en pesos; aquí se muestran en millones.

In [ ]:
import pandas as pd

p = requests.get(f"{BASE}/pronostico", params={"serie": "OPERATIVO_TOTAL", "modelo": "ETS", "horizonte": 5}).json()
tabla = pd.DataFrame([{
    "fecha": d["fecha"], "P10": d["p10"] / 1e6, "P50": d["p50"] / 1e6, "P90": d["p90"] / 1e6,
    "prob. déficit (%)": d["prob_deficit"] * 100, "alerta": d["alerta"]["nivel"],
    "MAE histórico": d["confianza"]["mae"] / 1e6, "cobertura histórica (%)": d["confianza"]["cobertura"],
} for d in p["dias"]]).set_index("fecha")
display(tabla.round(2))

a = p["alerta_deficit"]
print(f"Alerta de déficit: {a['nivel']} ({a['dias_en_alerta']} de {len(p['dias'])} días con P10 < 0); "
      f"colchón sugerido {a['colchon_sugerido'] / 1e6:.2f} M MXN")
print(f"Confianza: {p['confianza']['nivel']}, cumple {p['confianza']['umbrales_cumplidos']} de "
      f"{p['confianza']['umbrales_total']} umbrales; latencia del pronóstico {p['latencia_ms']:.0f} ms")

In [ ]:
m = requests.get(f"{BASE}/metricas", params={"modelo": "ETS", "serie": "OPERATIVO_TOTAL"}).json()
display(pd.DataFrame(m["umbrales"])[["nombre", "valor", "condicion", "unidad", "cumple"]])
display(pd.DataFrame(m["reglas_deficit"]).round(3))
display(pd.DataFrame(requests.get(f"{BASE}/alertas").json())[["serie", "nivel", "dias_en_alerta", "peor_p10", "probabilidad_maxima"]])

Otro sistema (por ejemplo, el ERP) también puede mandar su propia serie diaria y recibir el pronóstico con el mismo formato. Esta petición no usa el Excel cargado y por eso no trae métricas de backtest.

In [ ]:
historia = [{"fecha": f.strftime("%Y-%m-%d"), "valor": float(v)}
            for f, v in zip(pd.bdate_range("2026-08-03", periods=40), 1.5e6 + 4e5 * pd.Series(range(40)).mod(5))]
r = requests.post(f"{BASE}/pronostico", json={"serie_id": "caja-erp", "modelo": "ETS", "horizonte": 5, "historia": historia})
print(r.status_code)
display(pd.DataFrame(r.json()["dias"])[["fecha", "p10", "p50", "p90", "prob_deficit"]].round(0))

La API también valida un JSON sin cargarlo. Este tiene un movimiento sin concepto y otro con la fecha y el monto mal escritos:

In [ ]:
malo = {"movimientos": [
    {"fecha": "2026-09-01", "monto": 10.0, "banco": "BBVA"},
    {"fecha": "01/09/2026", "monto": "10", "banco": "BBVA", "concepto": "SPEI RECIBIDO"},
]}
v = requests.post(f"{BASE}/datos/validar", json=malo).json()
print("válido:", v["valido"])
display(pd.DataFrame(v["errores"]))

## Salida en archivo JSON

Al arrancar, la API escribió el pronóstico completo del modelo recomendado en `salidas/`. Con `POST /api/v1/reporte` se genera el de cualquier otro modelo. El archivo tiene, para cada serie, los P10, P50 y P90 de cada día, la alerta de déficit con su colchón sugerido y la confianza del modelo.

In [ ]:
import json

for _ in range(60):
    archivo = requests.get(f"{BASE}/salud").json()["archivo_salida"]
    if archivo:
        break
    time.sleep(1)
print("Archivo escrito al arrancar:", archivo)
otro = requests.post(f"{BASE}/reporte", params={"modelo": "SARIMAX"}).json()
print("Archivo nuevo:", otro["archivo"], f"({otro['bytes']:,} bytes)")

reporte = json.loads((RAIZ / archivo).read_text(encoding="utf-8"))
print(f"\n{reporte['modelo']}: confianza {reporte['confianza']['nivel']}, datos hasta {reporte['datos']['hasta']}")
display(pd.DataFrame([{
    "serie": s["serie"], "alerta": s["alerta_deficit"]["nivel"],
    "P50 acumulado (M)": s["acumulado"]["p50"] / 1e6, "peor P10 (M)": s["alerta_deficit"]["peor_p10"] / 1e6,
    "colchón sugerido (M)": s["alerta_deficit"]["colchon_sugerido"] / 1e6,
} for s in reporte["series"]]).round(2))

In [ ]:
# Descargar el archivo a tu computadora (no instala nada; es una descarga del navegador)
from google.colab import files
files.download(str(RAIZ / archivo))

## Abrir el dashboard

Colab publica el puerto del servidor detrás de su propio proxy. El dashboard usa rutas relativas (`api/v1/...`), así que funciona igual dentro del proxy. La primera celda lo muestra aquí mismo; la segunda da un enlace para abrirlo en otra pestaña. En el dashboard, "Descargar" baja el JSON del modelo elegido y "Guardar en el servidor" lo escribe en `salidas/`. El contrato de la API (OpenAPI) queda en `/docs`.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(PUERTO, path="/", height=1100)

In [ ]:
output.serve_kernel_port_as_window(PUERTO, path="/", anchor_text="Abrir el dashboard en otra pestaña")
output.serve_kernel_port_as_window(PUERTO, path="/docs", anchor_text="Abrir el contrato de la API (/docs)")

## Pruebas de validación y de contrato

Comprueban que la validación detecta cada dato que falta, que cada endpoint responde con el esquema acordado, que el archivo JSON de salida es idéntico a la respuesta de la API, que los cuantiles son los mismos que entrega el paquete de modelos, que las métricas reproducen el backtest de `results/` y que los errores llegan con un mensaje claro. La prueba en navegador (`tests/test_e2e.py`) se omite si Playwright no está instalado.

In [ ]:
!python -m pytest tests/test_validacion.py tests/test_alertas.py tests/test_api.py tests/test_pagina.py -q

## Actualizar la página publicada

La página de GitHub Pages no tiene servidor: muestra las respuestas de la API guardadas en la carpeta `docs/`. Cuando cambien los movimientos, esta celda vuelve a generar `docs/` y la descarga en un zip; después se sube a GitHub como se explica en `GUIA_GITHUB.md`.

In [ ]:
!python -m api.publicar --datos {ARCHIVO_DATOS}
!rm -f docs.zip && zip -qr docs.zip docs
files.download(str(RAIZ / "docs.zip"))

In [ ]:
# Detener la API al terminar
servidor.terminate()
servidor.wait(timeout=10)
print("API detenida")